In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
SOURCE_TABLE = "fuel_project_dev.bronze.fuel_transactions"
TARGET_TABLE = "fuel_project_dev.silver.fuel_transactions"
QUARANTINE_TABLE = "fuel_project_dev.quarantine.fuel_transactions"
CHECKPOINT_VOLUME = "/Volumes/fuel_project_dev/checkpoints/silver/fuel_transactions"
QUARANTINE_VOLUME = "/Volumes/fuel_project_dev/checkpoints/silver/fuel_transactions_quarantine"

In [0]:
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
  fill_id STRING,
  station_id STRING,
  start_time TIMESTAMP,
  end_time TIMESTAMP,
  fuel_type STRING,
  fuel_volume DOUBLE,
  payment_type STRING,
  _silver_ingestion_ts TIMESTAMP
)
USING DELTA
CLUSTER BY (station_id)
    TBLPROPERTIES (
        delta.autoOptimize.optimizeWrite = true,
        delta.autoOptimize.autoCompact = true
    )
    """
)

In [0]:
# Spark Structured Streaming
df = (
    spark.readStream
        .format("delta")
        .table(SOURCE_TABLE)
)

In [0]:
df = (
    df
        .withColumn("fill_id", col("FillID"))
        .withColumn("station_id", col("StationID"))
        .withColumn("start_time", to_timestamp("StartTime"))
        .withColumn("end_time", to_timestamp("EndTime"))
        .withColumn("fuel_type", col("FuelType"))
        .withColumn("fuel_volume", col("FuelVolume").cast("double"))
        .withColumn("payment_type", col("Payment"))
        .withColumn("_silver_ingestion_ts", current_timestamp())
)


In [0]:
df = (
    df.select("fill_id","station_id","start_time","end_time","fuel_type","fuel_volume","payment_type","_silver_ingestion_ts")
)

In [0]:
valid_payment_values = [
    "Cash",
    "Credit Card",
    "Debit Card",
    "UPI",
    "Multiple"
]

valid_df = df.filter(
    col("payment_type").isin(valid_payment_values)
)

invalid_payment_df = df.filter(
    col("payment_type").isin(valid_payment_values)
)

In [0]:
df = (
    valid_df
        .withWatermark("start_time", "10 minutes")  # state cleanup only
        .dropDuplicates(["fill_id"])
)

In [0]:
(
    invalid_payment_df
        .writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            CHECKPOINT_VOLUME
        )
        .trigger(processingTime="1 minute")
        .table(TARGET_TABLE)
)


In [0]:
(
    invalid_payment_df
        .withColumn("quarantine_reason", lit("INVALID_PAYMENT_TYPE"))
        .withColumn("_quarantined_ts", current_timestamp())
        .writeStream
        .format("delta")
        .outputMode("append")
        .trigger(processingTime="1 minute")
        .option(
            "checkpointLocation",
            QUARANTINE_VOLUME
        )
        .table(QUARANTINE_TABLE)
)

In [0]:
spark.streams.active